In [ ]:
#pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import medmnist
from medmnist import PneumoniaMNIST

import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score

In [ ]:
# Transformation for converting to PyTorch tensors
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


In [ ]:
# 1. Download and load the datasets
train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
val_dataset = PneumoniaMNIST(split='val', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)



100%|██████████| 4.17M/4.17M [00:02<00:00, 1.89MB/s]


In [ ]:
# 2. Wrap in PyTorch DataLoaders for CNN model
batch_size = 128
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# Checking datasets
print(("Training Dataset:"), len(train_dataset))
print(("Validation Dataset:"),len(val_dataset))
print(("Test Dataset:"),len(test_dataset))

Training Dataset: 4708
Validation Dataset: 524
Test Dataset: 624


In [ ]:
# CNN is the deep learning model of choice and 2D will be used instead of 1D since we are dealing with images and not sequences
class PneumoniaCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128*3*3,128),
            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(128,1)
        )

    def forward(self,x):

        x=self.features(x)
        x=self.classifier(x)

        return x

In [ ]:
# initializing the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PneumoniaCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
# training loop
epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images,labels in train_loader:

        images = images.to(device)

        labels = labels.float().to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}")

    print("Training Loss:",
          running_loss/len(train_loader))

Epoch 1/10
Training Loss: 0.4541177532157382
Epoch 2/10
Training Loss: 0.21925154207526026
Epoch 3/10
Training Loss: 0.1757580168746613
Epoch 4/10
Training Loss: 0.14292752178939613
Epoch 5/10
Training Loss: 0.13017260001317874
Epoch 6/10
Training Loss: 0.12054563978233852
Epoch 7/10
Training Loss: 0.10551104734878282
Epoch 8/10
Training Loss: 0.09014265682246234
Epoch 9/10
Training Loss: 0.09266721175329105
Epoch 10/10
Training Loss: 0.07564810149975724


In [ ]:
# validation
model.eval()

predictions=[]
truth=[]

with torch.no_grad():

    for images,labels in val_loader:

        images=images.to(device)

        outputs=model(images)

        preds=(torch.sigmoid(outputs)>0.5).cpu().numpy()

        predictions.extend(preds.flatten())

        truth.extend(labels.numpy().flatten())

val_accuracy = accuracy_score(
    truth,
    predictions
)

print("Validation Accuracy:",val_accuracy)

Validation Accuracy: 0.9599236641221374


In [ ]:
# test set evaluation
model.eval()

predictions=[]
truth=[]

with torch.no_grad():

    for images,labels in test_loader:

        images=images.to(device)

        outputs=model(images)

        preds=(torch.sigmoid(outputs)>0.5).cpu().numpy()

        predictions.extend(preds.flatten())

        truth.extend(labels.numpy().flatten())

test_accuracy = accuracy_score(
    truth,
    predictions
)

print("Test Accuracy:",test_accuracy)

Test Accuracy: 0.842948717948718


In [ ]:
# report for classification
from sklearn.metrics import classification_report

print(classification_report(
    truth,
    predictions,
    target_names=["Normal","Pneumonia"]
))

              precision    recall  f1-score   support

      Normal       0.97      0.60      0.74       234
   Pneumonia       0.80      0.99      0.89       390

    accuracy                           0.84       624
   macro avg       0.89      0.79      0.81       624
weighted avg       0.87      0.84      0.83       624

